# RFM — Quantile, Hybrid và KMeans (notebook chung)

Một input, một cohort HasEvent, một protocol để so **333/555, bốn hybrid và KMeans k=3–8**. Gom bucket4–10 nhóm là tùy chọn, mặc định tắt. Input: output notebook01, không cần notebook02.

- Fit biên/scaler/tâm tại2015-11; forecast cutoffDec2015–Apr2016 → acquisitionJan–May2016.
- Rate nhóm khởi tạo chỉ từNov→Dec; nhãn này có tại cutoffDec. Chấm trước, cập nhật sau, smoothing100 giống nhau cho mọi phương án. M-only giữ điểm upstream/100.
- Geometry: log1p+StandardScaler chung, cùng cohort hiện tại, silhouette cùng mẫu; CH/DB trên toàn cohort. Transition giao hai snapshot hiện tại, future coverage báo riêng.
- Top10%: kỳ vọng lựa chọn ngẫu nhiên trong nhóm đồng điểm; không lệ thuộc customer_id.
- NoObservedEvent không gán R giả, báo baseline riêng. Các tháng đã xem là development backtest; chưa chứng minh mô hình thắng trên giai đoạn mới. M upstream phải dùng nguồn đã biết tại cutoff.

Chưa chạy Santander sau hợp nhất. Chạy Run all trên Kaggle; cần numpy/pandas/scipy/scikit-learn/matplotlib/pyarrow/joblib.

Bổ sung sensitivity biênR/F và audit heterogeneity trongbucket; bản mới18 cell, chưa chạy lại Santander. Các kết quả cũ đã xóa để tránh trộn protocol/code.


In [ ]:
RUN_NAME = '03-rfm-method-comparison'

from pathlib import Path
import json, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, kmeans_plusplus
from sklearn.metrics import (silhouette_score, calinski_harabasz_score,
    davies_bouldin_score, adjusted_rand_score, average_precision_score, roc_auc_score)
INPUT_DIR = '/kaggle/input/notebooks/thientrantdt/01-r-f-m-analysis' # folder output notebook 01; None = auto-find in /kaggle/input
FIT_MONTH = '2015-11'
EVAL_MONTHS = ['2015-12','2016-01','2016-02','2016-03','2016-04']
FEATURES = ['R','F_event_months','M_group_points']
SEED = 42
SIL_SAMPLE = 2500 # O(sample^2); same customer sample across methods
MIN_GROUP_N = 100 # flag only; does not change assignments
MIN_HISTORY_N = 1000 # audit only; keep original M score
SAVE_ASSIGNMENTS = True
PREPROCESS = 'log1p' # 'raw' hoặc 'log1p'; giống nhau giữa hai notebook
RATE_PRIOR_N = 100 # smoothing cluster response, KHÔNG sửa feature M
# Base chosen for merge, not automatically optimized on evaluation.
MERGE_BASE='Hybrid_R4_F3_M5'
RUN_MERGE=False
VIEW_METHOD='Hybrid_R4_F3_M5' # or Merged_Profile_6 / Merged_ResponseW0.5_6
SHOW_DETAILS=False
K_OPTIONS=[3,4,5,6,7,8]
MIN_GROUP_EVENTS=20 # reliability flag, proposed threshold
MAX_WEAK_CUSTOMER_SHARE=.05
MIN_MERGED_SHARE=.02
OUTPUT_DIR=Path('/kaggle/working')/RUN_NAME
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
pd.set_option('display.max_columns',20)
plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False})


In [ ]:
root = Path(INPUT_DIR) if INPUT_DIR else Path('/kaggle/input')
candidates = list(root.rglob('rfm_all_months/manifest.json'))
if root.name == 'rfm_all_months' and (root/'manifest.json').exists():
    candidates = [root/'manifest.json']
assert len(candidates)==1, f'Found {len(candidates)} manifests; set INPUT_DIR. {candidates}'
feature_dir = candidates[0].parent
manifest = json.loads(candidates[0].read_text())
files = {pd.Period(Path(f).stem.removeprefix('rfm_'),freq='M'):feature_dir/f
         for f in manifest['feature_files']}
event_paths = list(feature_dir.parent.rglob('acquisition_events.parquet'))
assert len(event_paths)==1, 'Need acquisition_events.parquet from notebook 01'
events = pd.read_parquet(event_paths[0], columns=['customer_id','record_date'])
events['month'] = pd.to_datetime(events.record_date).dt.to_period('M')
responders = {m:pd.Index(g.customer_id.unique()) for m,g in events.groupby('month')}
del events
fit_month = pd.Period(FIT_MONTH, freq='M')
eval_months = [pd.Period(m,freq='M') for m in EVAL_MONTHS]
assert all(m > fit_month and m in files and m+1 in files for m in eval_months)
assert fit_month in files and fit_month+1 <= min(eval_months)
def load_month(month):
    p = pd.read_parquet(files[month])
    assert p.customer_id.is_unique
    assert p.month_idx.eq(month.year*12+month.month).all()
    ok = p.has_event.eq(True) & p.products_complete.eq(True)
    ok &= np.isfinite(p[FEATURES].to_numpy(dtype=float)).all(axis=1)
    q = p.loc[ok].copy().reset_index(drop=True)
    assert len(q)>1 and q.R.ge(0).all() and q.F_event_months.ge(1).all()
    assert q.M_group_points.between(0,100).all(), 'M outside [0,100]'
    return q, {'month':str(month),'all_n':len(p),'has_event_n':int(p.has_event.sum()),
               'clustered_n':len(q),'excluded_n':len(p)-len(q)}
def next_target(p, month):
    nxt = pd.read_parquet(files[month+1], columns=['customer_id','products_complete'])
    eligible = p.customer_id.isin(nxt.loc[nxt.products_complete.eq(True),'customer_id']).to_numpy()
    y = p.customer_id.isin(responders.get(month+1,pd.Index([]))).astype(int).to_numpy()
    return eligible,y
fit, fit_audit = load_month(fit_month)
assert fit[FEATURES].ge(0).all().all(), 'Negative features: check data contract'
assert fit.M_group_points.le(100).all(), 'Expected M points in [0,100]'
def raw_transform(p):
    z=p[FEATURES].to_numpy(float)
    if not np.isfinite(z).all() or (z<0).any(): raise ValueError('Invalid RFM values')
    return np.log1p(z) if PREPROCESS=='log1p' else z
assert PREPROCESS in ['raw','log1p']
scaler = StandardScaler().fit(raw_transform(fit))
def transform(p): return scaler.transform(raw_transform(p))
Xfit = transform(fit)
print('Fit:',fit_month,'HasEvent eligible:',len(fit),'Eval:',EVAL_MONTHS)
display(fit[FEATURES].describe().round(2))


In [ ]:
def sample_indices(n):
    return np.sort(np.random.default_rng(SEED).choice(n,min(n,SIL_SAMPLE),replace=False))
def quality(X, labels, idx):
    labs=np.asarray(labels); k=len(np.unique(labs)); sub=labs[idx]
    out={'actual_groups':k,'silhouette':np.nan,'CH':np.nan,'DB':np.nan,
         'silhouette_sample_n':len(idx),'sample_groups':len(np.unique(sub))}
    if 1 < len(np.unique(sub)) < len(sub): out['silhouette']=silhouette_score(X[idx],sub)
    if 1 < k < len(labs):
        out['CH']=calinski_harabasz_score(X,labs); out['DB']=davies_bouldin_score(X,labs)
    counts=pd.Series(labs).value_counts()
    out.update(min_group_n=counts.min(),max_group_share=counts.max()/len(labs),
               small_group_customer_share=counts[counts<MIN_GROUP_N].sum()/len(labs))
    return out

def metrics(y, score):
    y=np.asarray(y); score=np.asarray(score); base=y.mean()
    # Expected precision under random allocation inside the cutoff tie.
    # All equally scored customers get equal fractional selection weight.
    n_top=max(1,int(.1*len(y))); cutoff=np.partition(score,len(score)-n_top)[len(score)-n_top]
    above=score>cutoff; tied=score==cutoff
    top_precision=(y[above].sum()+(n_top-above.sum())*y[tied].mean())/n_top
    return {'AP':average_precision_score(y,score) if y.sum() else np.nan,
            'AUC':roc_auc_score(y,score) if len(np.unique(y))==2 else np.nan,
            'Precision10':top_precision,'Lift10':top_precision/base if base else np.nan,
            'Brier':np.mean((score-y)**2),'baseline':base}

def profile(p, labels):
    z=p[FEATURES].copy(); z['cluster']=np.asarray(labels)
    t=z.groupby('cluster').agg(n=('R','size'),R_median=('R','median'),
        F_median=('F_event_months','median'),M_median=('M_group_points','median'))
    for f in FEATURES:
        t[f+'_q25']=z.groupby('cluster')[f].quantile(.25)
        t[f+'_q75']=z.groupby('cluster')[f].quantile(.75)
    return t.reset_index()

def wilson(k,n):
    p=k/n; d=1+1.96**2/n
    center=(p+1.96**2/(2*n))/d
    half=1.96*np.sqrt(p*(1-p)/n+1.96**2/(4*n*n))/d
    return center-half,center+half


## 1. Biên và ý nghĩa

R là thời gian từ event, không mặc định R gần luôn có rate cao hơn. Không acquisition thì R tăng; có event thì về0 (khi quan sát hợp lệ). F tích lũy phụ thuộc exposure; mức nguyên giữ khách cùng F chung nhóm. M là propensity score, không phải tiền; M có thể đổi do cập nhật score table.

Quantile giữ ties nên số mức thực tế có thể giảm. Các ngưỡng R/F là ứng viên nghiệp vụ, không phải kết luận tối ưu. Baseline555 vẫn giữ để so.


**Thử thêm có kiểm soát:** REDA=`0 / 1–2 / 3–7 / ≥8`; F4=`1 / 2 / 3–4 / ≥5`. So R-only change, F-only change và cả hai, giữ cùng M5 edges. Số ô tối đa lần lượt60/60/80 (giả sửM có5 mức), so45 ô gốc. Các biên này đề xuất từ EDA đã xem, nên kết quả vẫn là development evidence. Không đưa NoObservedEvent vàoR=0. Chưa tự đổi MERGE_BASE hoặc VIEW_METHOD theo kết quả.


In [ ]:
specs={};models={};boundary_rows=[]
def qedges(v,q):
    if v.nunique()<2:return np.array([-np.inf,np.inf])
    _,edges=pd.qcut(v,q,retbins=True,duplicates='drop')
    edges=np.array(edges,float);edges[0]=-np.inf;edges[-1]=np.inf
    return edges
for q in [3,5]:
    specs[f'Quantile_{q}{q}{q}']={f:qedges(fit[f],q) for f in FEATURES}
for label,redges in [('R4',[-np.inf,1,3,np.inf]),('R6',[-np.inf,1,5,np.inf])]:
    for q in [3,5]:
        specs[f'Hybrid_{label}_F3_M{q}']={'R':np.array(redges,float),
            'F_event_months':np.array([-np.inf,1,2,np.inf]),'M_group_points':qedges(fit.M_group_points,q)}
# Controlled boundary sensitivity: same M5 edges, only R/F boundaries differ.
M5_EDGES=specs['Hybrid_R4_F3_M5']['M_group_points'].copy()
for name,r_edges,f_edges in [
    ('Hybrid_REDA_F3_M5',[-np.inf,0,2,7,np.inf],[-np.inf,1,2,np.inf]),
    ('Hybrid_R4_F4_M5',[-np.inf,1,3,np.inf],[-np.inf,1,2,4,np.inf]),
    ('Hybrid_REDA_F4_M5',[-np.inf,0,2,7,np.inf],[-np.inf,1,2,4,np.inf])]:
    specs[name]={'R':np.array(r_edges,float),'F_event_months':np.array(f_edges,float),
                 'M_group_points':M5_EDGES.copy()}
for name,edges in specs.items():
    models[name]=edges
    for f,e in edges.items():
        counts=pd.cut(fit[f],e,labels=False).value_counts()
        for j in range(len(e)-1):boundary_rows.append(dict(method=name,feature=f,level=j+1,
            lower_exclusive=e[j],upper_inclusive=e[j+1],fit_n=int(counts.get(j,0)),actual_levels=len(e)-1))
def assign(name,p):
    levels=[(pd.cut(p[f],specs[name][f],labels=False,include_lowest=True).astype(int)+1).astype(str) for f in FEATURES]
    return ('R'+levels[0]+'_F'+levels[1]+'_M'+levels[2]).to_numpy()
boundaries=pd.DataFrame(boundary_rows)
boundaries.to_csv(OUTPUT_DIR/'bucket_boundaries.csv',index=False)
display(boundaries[boundaries.method.eq(VIEW_METHOD)])
display(pd.DataFrame({'feature':FEATURES,'unique':[fit[f].nunique() for f in FEATURES],
    'largest_tie_share':[fit[f].value_counts(normalize=True).iloc[0] for f in FEATURES]}))


## 2. Fit các phương án (gom bucket tùy chọn)

KMeans fit từ cùng Xfit; n_init10/max_iter300/seed42. C1…Ck là nominal IDs cố định theo tâm, không phải ranking acquisition. Fit silhouette chỉ chọn một ứng viên trước evaluation; không bắt buộc chọn winner theo geometry.

Gom bucket dùng MERGE_BASE=hybridR4/F3/M5 khi RUN_MERGE=True. Profile chỉ dùng RFM; Response thêm nhãnNov→Dec, chỉ có từ cuốiDec. Rule là OR các ô; weighted Ward greedy không bảo đảm tối ưu toàn cục. Nhãn “Merged” không có cùng mapping như các notebook cũ dùngR6/555.


In [ ]:
merge_maps={}
BASE_BUCKET_METHOD=MERGE_BASE
if RUN_MERGE:
    from itertools import product, combinations
    BASE_BUCKET_METHOD = MERGE_BASE
    MERGE_K_VALUES = list(range(4,11))
    # 0: RFM profiles only; .5: additionally preserve historical response similarity.
    # Response labels are fit_month -> fit_month+1, available at first evaluation cutoff.
    MERGE_RATE_WEIGHTS = [0.0, 0.5]
    CELL_RATE_PRIOR_N = 100
    original_assign = assign
    fit_cells = original_assign(BASE_BUCKET_METHOD,fit)
    levels_by_cell = {f'R{r}_F{f}_M{m}':(r,f,m) for r,f,m in product(*[
        range(1,len(specs[BASE_BUCKET_METHOD][feature])) for feature in FEATURES])}
    all_cells=sorted(levels_by_cell)
    cell_frame=pd.DataFrame(Xfit,columns=['zR','zF','zM'])
    cell_frame['cell']=fit_cells
    cell_stats=cell_frame.groupby('cell').agg(n=('zR','size'),zR=('zR','mean'),zF=('zF','mean'),zM=('zM','mean'))
    observed=sorted(cell_stats.index)
    assert len(observed)>=max(MERGE_K_VALUES), 'Too few occupied base cells; reduce MERGE_K_VALUES'
    mask0,y0=next_target(fit,fit_month)
    assert mask0.any()
    ct=pd.DataFrame({'cell':fit_cells[mask0],'y':y0[mask0]}).groupby('cell').y.agg(label_n='size',events='sum')
    cell_stats=cell_stats.join(ct).fillna({'label_n':0,'events':0})
    base_rate=y0[mask0].mean()
    cell_stats['historical_rate']=(cell_stats.events+CELL_RATE_PRIOR_N*base_rate)/(cell_stats.label_n+CELL_RATE_PRIOR_N)
    # Empty fit cells get a deterministic nearest occupied-cell fallback by ordinal grid distance.
    # This fallback is frozen; its share is audited in every later month.
    empty_cells=set(all_cells)-set(observed)
    fallback={cell:min(observed,key=lambda b:(sum(abs(a-c) for a,c in zip(levels_by_cell[cell],levels_by_cell[b])),b))
              for cell in empty_cells}
    merge_maps={}; merge_logs=[]; rule_rows=[]; bridge_rows=[]
    for weight in MERGE_RATE_WEIGHTS:
        mode='Profile' if weight==0 else f'ResponseW{weight:g}'
        Z=cell_stats.loc[observed,['zR','zF','zM']].to_numpy()
        ns=cell_stats.loc[observed,'n'].to_numpy(float)
        if weight>0:
            rate=np.clip(cell_stats.loc[observed,'historical_rate'].to_numpy(),1e-5,1-1e-5)
            logits=np.log(rate/(1-rate));mu=np.average(logits,weights=ns)
            sd=np.sqrt(np.average((logits-mu)**2,weights=ns))
            Z=np.column_stack([Z,weight*(logits-mu)/max(sd,1e-8)])
        groups={i:{'cells':{c},'n':ns[i],'center':Z[i]} for i,c in enumerate(observed)}
        neighbors={i:set() for i in groups}
        for i,j in combinations(groups,2):
            if sum(abs(a-b) for a,b in zip(levels_by_cell[observed[i]],levels_by_cell[observed[j]]))==1:
                neighbors[i].add(j);neighbors[j].add(i)
        next_id=len(groups)
        while len(groups)>min(MERGE_K_VALUES):
            pairs=[(i,j) for i in sorted(groups) for j in sorted(neighbors[i]) if i<j]
            bridged=not pairs
            if bridged:pairs=list(combinations(sorted(groups),2))
            def cost(pair):
                a,b=(groups[i] for i in pair)
                return float(a['n']*b['n']/(a['n']+b['n'])*np.sum((a['center']-b['center'])**2))
            i,j=min(pairs,key=lambda pair:(cost(pair),pair))
            a,b=groups[i],groups[j];delta=cost((i,j))
            merge_logs.append({'mode':mode,'groups_before':len(groups),'groups_after':len(groups)-1,
                'weighted_SSE_increase':delta,'bridged_disconnected_graph':bridged})
            if bridged:bridge_rows.append({'mode':mode,'groups_before':len(groups),'left':sorted(a['cells']),'right':sorted(b['cells'])})
            adj=(neighbors[i]|neighbors[j])-{i,j}
            groups[next_id]={'cells':a['cells']|b['cells'],'n':a['n']+b['n'],
                'center':(a['center']*a['n']+b['center']*b['n'])/(a['n']+b['n'])}
            for h in adj:neighbors[h].difference_update({i,j});neighbors[h].add(next_id)
            del groups[i],groups[j],neighbors[i],neighbors[j]
            neighbors[next_id]=adj;next_id+=1
            k=len(groups)
            if k in MERGE_K_VALUES:
                name=f'Merged_{mode}_{k}';mapping={}
                # IDs are descriptive nominal labels, NOT a purchase ranking.
                for gid,g in enumerate(sorted(groups.values(),key=lambda g:min(g['cells'])),1):
                    for cell in sorted(g['cells']):mapping[cell]=f'G{gid:02d}'
                for cell,near in fallback.items():mapping[cell]=mapping[near]
                merge_maps[name]=mapping
                models[name]={'kind':'merged_quantile','base_method':BASE_BUCKET_METHOD,
                    'cell_to_group':mapping,'rate_weight':weight,'learned_by_month':str(fit_month+1 if weight else fit_month)}
                for cell,group in sorted(mapping.items()):
                    rule_rows.append({'method':name,'group':group,'cell':cell,'fit_empty':cell in empty_cells,
                        'fallback_cell':fallback.get(cell,''),'fit_n':int(cell_stats.loc[cell,'n']) if cell in cell_stats.index else 0})
    def assign(name,p):
        if name in merge_maps:
            cells=original_assign(BASE_BUCKET_METHOD,p)
            return np.array([merge_maps[name][cell] for cell in cells])
        return original_assign(name,p)
    merge_history=pd.DataFrame(merge_logs); rules=pd.DataFrame(rule_rows)
    rules.to_csv(OUTPUT_DIR/'merged_exact_cell_rules.csv',index=False)
    merge_history.to_csv(OUTPUT_DIR/'merge_history.csv',index=False)
    cell_stats.to_csv(OUTPUT_DIR/'merge_fit_cell_profiles.csv')
    (OUTPUT_DIR/'merge_maps.json').write_text(json.dumps({'base_method':BASE_BUCKET_METHOD,
        'maps':merge_maps,'empty_cell_fallback':fallback,'weights':MERGE_RATE_WEIGHTS},indent=2))
    print('Merged candidates:',len(merge_maps))
    if bridge_rows:print('WARNING: disconnected grid bridged:',bridge_rows)

# KMeans dispatch: the exact same evaluation loop below scores all methods.
bucket_assign=assign
kmeans_models={};kmeans_label_maps={};fit_geometry_rows=[];center_rows=[]
fit_ix=sample_indices(len(fit))
for k in K_OPTIONS:
    name=f'KMeans_{k}'
    model=KMeans(n_clusters=k,n_init=10,max_iter=300,random_state=SEED)
    labels=model.fit_predict(Xfit)
    centers_raw=np.expm1(scaler.inverse_transform(model.cluster_centers_))
    order=np.lexsort((centers_raw[:,0],centers_raw[:,1],centers_raw[:,2]))
    mapping={int(original):f'C{j+1}' for j,original in enumerate(order)}
    kmeans_models[name]=model;kmeans_label_maps[name]=mapping
    models[name]={'kind':'kmeans','k':k,'model':model,'label_map':mapping,'learned_by_month':str(fit_month)}
    nominal=np.array([mapping[int(x)] for x in labels])
    fit_geometry_rows.append(dict(method=name,n_iter=int(model.n_iter_),inertia=float(model.inertia_),
        **quality(Xfit,nominal,fit_ix)))
    for j,center in enumerate(centers_raw):
        center_rows.append(dict(method=name,cluster=mapping[j],**dict(zip(FEATURES,center))))
def assign(name,p):
    if name in kmeans_models:
        lab=kmeans_models[name].predict(transform(p))
        return np.array([kmeans_label_maps[name][int(x)] for x in lab])
    return bucket_assign(name,p)
fit_geometry=pd.DataFrame(fit_geometry_rows)
fit_geometry.to_csv(OUTPUT_DIR/'kmeans_fit_geometry.csv',index=False)
pd.DataFrame(center_rows).to_csv(OUTPUT_DIR/'kmeans_centers_backtransformed.csv',index=False)
fit_k_candidate=fit_geometry.sort_values(['silhouette','method'],ascending=[False,True]).iloc[0].method
print('Fit-silhouette candidate (not final winner):',fit_k_candidate)
display(fit_geometry[['method','silhouette','DB','min_group_n','inertia','n_iter']].round(4))


## 3. Backtest và scorecard

AP/Lift đánh giá ranking; Brier đánh giá xác suất; silhouette chỉ mô tả geometry. Nhãn thiếu không thành0. Lift@10% của bảng phương án dùng baseline HasEvent; baselineAll được báo riêng. Small group threshold chỉ flag, không thay assignment. Chi tiết outcomes/profile/geometry xuất CSV thay vì in nhiều bảng.

KMeans và bucket có cùng train labels/smoothing/cohort/top10-ties. Các số cũ từ notebook KMeans chưa chuẩn hóa không được chép vào bảng này.


In [ ]:
histories={};geometry_rows=[];outcome_rows=[];score_rows=[];prof_rows=[];audit_rows=[]
mask,y=next_target(fit,fit_month)
assert mask.any()
for name in models:
    labs=assign(name,fit)
    histories[name]=pd.DataFrame({'cluster':labs[mask],'y':y[mask]}).groupby('cluster').y.agg(n='size',a='sum')
    prof_rows.append(profile(fit,labs).assign(method=name,month=str(fit_month)))
    if SAVE_ASSIGNMENTS:
        fit[['customer_id']+FEATURES].assign(cluster=labs,as_of_month=str(fit_month),method=name).to_parquet(OUTPUT_DIR/f'assignments_{name}_{fit_month}.parquet',index=False)
for month in pd.period_range(fit_month+1,max(eval_months),freq='M'):
    p,audit=load_month(month);mask,y=next_target(p,month);assert mask.any()
    is_eval=month in eval_months
    if is_eval:
        X=transform(p);ix=sample_indices(len(p))
        audit.update(outcome_n=int(mask.sum()),outcome_coverage=float(mask.mean()),
            M_low_support_share=float(p.M_group_history_n.lt(MIN_HISTORY_N).mean()) if 'M_group_history_n' in p else np.nan)
        audit_rows.append(audit)
    for name in models:
        labs=assign(name,p);h=histories[name];base=h.a.sum()/h.n.sum()
        rate=(h.a+RATE_PRIOR_N*base)/(h.n+RATE_PRIOR_N)
        scores=pd.Series(labs[mask]).map(rate).fillna(base).to_numpy()
        counts=pd.DataFrame({'cluster':labs[mask],'y':y[mask]}).groupby('cluster').y.agg(n='size',a='sum')
        if is_eval:
            score_rows.append(dict(method=name,month=str(month),n=int(mask.sum()),**metrics(y[mask],scores)))
            geometry_rows.append(dict(method=name,month=str(month),**quality(X,labs,ix)))
            prof_rows.append(profile(p,labs).assign(method=name,month=str(month)))
            t=counts.reset_index();t['rate']=t.a/t.n;t['lift']=t.rate/y[mask].mean() if y[mask].mean() else np.nan
            t['ci_low'],t['ci_high']=wilson(t.a,t.n);outcome_rows.append(t.assign(method=name,month=str(month)))
            if SAVE_ASSIGNMENTS:
                p[['customer_id']+FEATURES].assign(cluster=labs,as_of_month=str(month),method=name).to_parquet(OUTPUT_DIR/f'assignments_{name}_{month}.parquet',index=False)
        histories[name]=h.add(counts,fill_value=0)
    if is_eval:score_rows.append(dict(method='M_only',month=str(month),n=int(mask.sum()),
        **metrics(y[mask],p.loc[mask,'M_group_points'].to_numpy()/100)))
outcomes=pd.concat(outcome_rows,ignore_index=True);profiles=pd.concat(prof_rows,ignore_index=True)
geometry=pd.DataFrame(geometry_rows);predictive=pd.DataFrame(score_rows)
assert predictive.groupby('month').n.nunique().eq(1).all(), 'Different predictive cohorts'
assert not outcomes.duplicated(['method','month','cluster']).any()
for name,t in [('cluster_outcomes',outcomes),('profiles',profiles),('geometry',geometry),('predictive',predictive),('coverage_audit',pd.DataFrame(audit_rows))]:
    t.to_csv(OUTPUT_DIR/(name+'.csv'),index=False)
def rank_stability(method):
    t=outcomes[(outcomes.method==method)&outcomes.n.ge(MIN_GROUP_N)].pivot(index='cluster',columns='month',values='rate')
    corr=t.corr(method='spearman',min_periods=3).to_numpy();off=corr[np.triu_indices_from(corr,1)]
    return np.nanmin(off) if np.isfinite(off).any() else np.nan
summary=predictive.groupby('method').agg(AP=('AP','mean'),AP_worst=('AP','min'),Lift10=('Lift10','mean'),
    Lift10_worst=('Lift10','min'),Brier=('Brier','mean'),AUC=('AUC','mean'))
summary=summary.join(geometry.groupby('method').agg(groups=('actual_groups','max'),silhouette=('silhouette','mean'),small_share=('small_group_customer_share','max')))
summary['rank_stability_min']=[rank_stability(name) if name in models else np.nan for name in summary.index]
summary['AP_retained_555']=summary.AP/summary.loc['Quantile_555','AP']
summary['Lift_retained_555']=summary.Lift10/summary.loc['Quantile_555','Lift10']
summary.to_csv(OUTPUT_DIR/'decision_scorecard.csv')
display(summary.loc[list(specs)+list(kmeans_models)+['M_only']].round(4))
if RUN_MERGE:
    names=[f'Merged_{b}_{k}' for b in ['Profile','ResponseW0.5'] for k in [6,10]]
    display(summary.loc[names].round(4))
fig,axes=plt.subplots(1,2,figsize=(13,4))
for name,g in predictive[predictive.method.isin(['Quantile_555',VIEW_METHOD,'KMeans_5','KMeans_6',fit_k_candidate,'M_only'])].groupby('method'):
    axes[0].plot(g.month,g.AP,'o-',label=name);axes[1].plot(g.month,g.Lift10,'o-',label=name)
axes[0].set_title('AP by month');axes[1].set_title('Lift@10% versus HasEvent')
for ax in axes:ax.tick_params(axis='x',rotation=35)
axes[1].legend(bbox_to_anchor=(1.02,1),fontsize=8);plt.tight_layout();plt.show()

# Does changing only R/F improve quality on identical monthly cohorts?
SENSITIVITY_METHODS=['Hybrid_R4_F3_M5','Hybrid_REDA_F3_M5','Hybrid_R4_F4_M5','Hybrid_REDA_F4_M5']
reference=predictive[predictive.method.eq('Hybrid_R4_F3_M5')].set_index('month')
sensitivity=predictive[predictive.method.isin(SENSITIVITY_METHODS)].copy()
for col in ['AP','Lift10','Brier']:
    sensitivity['delta_'+col]=sensitivity[col]-sensitivity.month.map(reference[col])
sensitivity.to_csv(OUTPUT_DIR/'boundary_sensitivity_monthly.csv',index=False)
sensitivity_summary=summary.loc[SENSITIVITY_METHODS,['groups','AP','AP_worst','Lift10','Lift10_worst','Brier','rank_stability_min']].copy()
sensitivity_summary=sensitivity_summary.join(sensitivity.groupby('method')[['delta_AP','delta_Lift10','delta_Brier']].mean())
display(sensitivity_summary.round(4));sensitivity_summary.to_csv(OUTPUT_DIR/'boundary_sensitivity_summary.csv')


In [ ]:
# Population baseline: same observable target, no invented R for NoEvent.
pop_rows=[]
for month in eval_months:
    p=pd.read_parquet(files[month]);p=p[p.products_complete.eq(True)].reset_index(drop=True)
    ok,y=next_target(p,month);p=p.loc[ok].copy();p['y']=y[ok];baseline=p.y.mean()
    for name,sub in [('All',p),('HasEvent',p[p.has_event.eq(True)]),('NoObservedEvent',p[p.has_event.eq(False)])]:
        if not len(sub):continue
        lo,hi=wilson(sub.y.sum(),len(sub))
        pop_rows.append(dict(month=str(month),cohort=name,n=len(sub),rate=sub.y.mean(),ci_low=lo,ci_high=hi,
            lift_all=sub.y.mean()/baseline if baseline else np.nan))
population=pd.DataFrame(pop_rows);population.to_csv(OUTPUT_DIR/'population_baselines.csv',index=False)
display(population.groupby('cohort').agg(months=('month','nunique'),rate_mean=('rate','mean'),rate_min=('rate','min'),rate_max=('rate','max')).round(4))


## 4. Đọc phương án và chuyển nhóm

Chỉnh VIEW_METHOD để xem hybrid hoặc phương án gom. Rate heatmap sắp theo trung bình để trình bày; ID không đổi theo ranking. Membership transition chỉ giao snapshot hiện tại; future outcome ở tháng sau đích, có coverage riêng. Không suy causal từ stay/move. R≥6 không tự trở thành churned; panel gap thuộc notebook lifecycle.


In [ ]:
assert VIEW_METHOD in models
v=outcomes[outcomes.method.eq(VIEW_METHOD)]
p0=profiles[(profiles.method==VIEW_METHOD)&(profiles.month==str(fit_month))].set_index('cluster')
t=v.groupby('cluster').agg(rate_mean=('rate','mean'),lift_mean=('lift','mean'),rate_min=('rate','min'),
    rate_max=('rate','max'),outcome_n_min=('n','min'))
t=p0[['n','R_median','F_median','M_median']].join(t);t['fit_share']=t.n/t.n.sum()
t=t.sort_values('rate_mean',ascending=False);display(t.round(4));t.to_csv(OUTPUT_DIR/'view_group_profiles.csv')
m=v.pivot(index='cluster',columns='month',values='rate').reindex(t.index)
fig,ax=plt.subplots(figsize=(8,max(4,.24*len(m))))
im=ax.imshow(m.to_numpy(float),aspect='auto',cmap='YlGnBu');fig.colorbar(im,ax=ax)
ax.set_xticks(range(len(m.columns)),m.columns,rotation=35);ax.set_yticks(range(len(m)),m.index,fontsize=7)
ax.set_title(VIEW_METHOD+' — acquisition t+1');plt.tight_layout();plt.show()
# Heterogeneity after merge: warn if constituent base cells differ.
if VIEW_METHOD in merge_maps:
    b=outcomes[outcomes.method.eq(MERGE_BASE)].copy();b['group']=b.cluster.map(merge_maps[VIEW_METHOD])
    b=b[b.n.ge(MIN_GROUP_N)]
    het=[]
    for (month,group),g in b.groupby(['month','group']):
        avg=np.average(g.rate,weights=g.n)
        het.append(dict(month=month,group=group,cells=len(g),n=int(g.n.sum()),
            within_sd=np.sqrt(np.average((g.rate-avg)**2,weights=g.n)),rate_min=g.rate.min(),rate_max=g.rate.max()))
    heterogeneity=pd.DataFrame(het);heterogeneity.to_csv(OUTPUT_DIR/'view_subcell_heterogeneity.csv',index=False)
    display(heterogeneity.groupby('group')[['cells','within_sd','rate_min','rate_max']].mean().round(4))


## 4.1. Bucket có che khác biệt bên trong không?

Đối với các hybridM5, tách từng ô theo **giá trị R nguyên** hoặc **F nguyên**, giữ cố định hai bucket còn lại. Chỉ tính dải/SD từ subgroups đủn/event; xuất cả dòng thiếu support để không che mất đuôi. SD là mô tả, còn nhiễu lấy mẫu; không phải kiểm định hay causal effect. CI Wilson tính riêng từng tháng. Chênh lệch không chứng minh cần tách nếu không bền qua thời gian.

Bảng chính: tỷ lệ khách thuộc subgroups yếu và độ phân tán trong bucket. Chi tiết raw R/F, n, events, rate và CI ở CSV. Nếu VIEW_METHOD làKMeans/merged, audit vẫn chạy cho bốn hybrid ứng viên thay vì diễn giải KMeans như bucket chữ nhật.


In [ ]:

subcell_rows=[]
for month in eval_months:
    p,_=load_month(month);ok,y=next_target(p,month)
    z=p.loc[ok,['customer_id']+FEATURES].copy().reset_index(drop=True);z['y']=y[ok]
    for method in SENSITIVITY_METHODS:
        labels=assign(method,z)
        for feature in ['R','F_event_months']:
            frame=pd.DataFrame({'cluster':labels,'raw_value':z[feature].to_numpy(),'y':z.y.to_numpy()})
            for (cluster,value),g in frame.groupby(['cluster','raw_value']):
                nn=len(g);aa=int(g.y.sum());lo,hi=wilson(aa,nn)
                subcell_rows.append(dict(method=method,month=str(month),cluster=cluster,feature=feature,
                    raw_value=value,n=nn,events=aa,rate=aa/nn,ci_low=lo,ci_high=hi,
                    supported=nn>=MIN_GROUP_N and aa>=MIN_GROUP_EVENTS))
raw_audit=pd.DataFrame(subcell_rows)
raw_audit.to_csv(OUTPUT_DIR/'within_bucket_raw_RF_monthly.csv',index=False)
spread_rows=[]
for key,g in raw_audit.groupby(['method','month','cluster','feature']):
    supported=g[g.supported];count=len(supported)
    avg=np.average(supported.rate,weights=supported.n) if count else np.nan
    spread_rows.append(dict(zip(['method','month','cluster','feature'],key),
        n=int(g.n.sum()),raw_levels=len(g),supported_levels=count,
        weak_share=g.loc[~g.supported,'n'].sum()/g.n.sum(),
        weighted_rate_sd=np.sqrt(np.average((supported.rate-avg)**2,weights=supported.n)) if count>=2 else np.nan,
        rate_range=supported.rate.max()-supported.rate.min() if count>=2 else np.nan))
spread=pd.DataFrame(spread_rows);spread.to_csv(OUTPUT_DIR/'within_bucket_spread_monthly.csv',index=False)
compact=[]
for (method,feature),g in spread.groupby(['method','feature']):
    valid=g.weighted_rate_sd.notna()
    compact.append(dict(method=method,feature=feature,
        weak_share=(g.weak_share*g.n).sum()/g.n.sum(),
        within_sd_weighted=np.average(g.loc[valid,'weighted_rate_sd'],weights=g.loc[valid,'n']) if valid.any() else np.nan,
        multi_supported_cell_months=int(valid.sum())))
compact=pd.DataFrame(compact);display(compact.round(4))
compact.to_csv(OUTPUT_DIR/'within_bucket_audit_summary.csv',index=False)
# Drill-down one stable parent bucket; inspect raw-value rate by month, not only pooled averages.
AUDIT_METHOD='Hybrid_R4_F3_M5';AUDIT_CELL='R3_F3_M5';AUDIT_FEATURE='F_event_months'
d=raw_audit[(raw_audit.method==AUDIT_METHOD)&(raw_audit.cluster==AUDIT_CELL)&
            (raw_audit.feature==AUDIT_FEATURE)&raw_audit.supported]
if len(d):
    fig,ax=plt.subplots(figsize=(9,4))
    for month,g in d.groupby('month'):
        g=g.sort_values('raw_value')
        ax.plot(g.raw_value,g.rate,'o-',label=month)
    ax.set(title=AUDIT_CELL+' — raw '+AUDIT_FEATURE,xlabel='Raw value',ylabel='Acquisition t+1 rate')
    ax.legend();plt.tight_layout();plt.show()
else:print('No supported raw subgroups for this drilldown; change AUDIT_CELL / feature.')


In [ ]:
transition_rows=[];months=[fit_month]+eval_months;last_matrix=None
for left,right in zip(months[:-1],months[1:]):
    if right.ordinal-left.ordinal!=1:continue
    p,_=load_month(left);q,_=load_month(right)
    a=pd.DataFrame({'customer_id':p.customer_id,'src':assign(VIEW_METHOD,p)})
    b=pd.DataFrame({'customer_id':q.customer_id,'dst':assign(VIEW_METHOD,q)})
    pair=a.merge(b,on='customer_id',validate='one_to_one')
    if right+1 in files:
        ok,y=next_target(q,right)
        future=pd.DataFrame({'customer_id':q.customer_id,'y':np.where(ok,y,np.nan)})
        pair=pair.merge(future,on='customer_id',validate='one_to_one')
    else:pair['y']=np.nan
    denom=pair.groupby('src').size()
    for (src,dst),g in pair.groupby(['src','dst']):
        nn=g.y.notna().sum();aa=g.y.sum();lo,hi=wilson(aa,nn) if nn else (np.nan,np.nan)
        transition_rows.append(dict(from_month=str(left),to_month=str(right),src=src,dst=dst,n=len(g),
            share_from=len(g)/denom[src],future_n=nn,future_coverage=nn/len(g),future_rate=aa/nn if nn else np.nan,
            ci_low=lo,ci_high=hi,stay=src==dst))
    last_matrix=pd.crosstab(pair.src,pair.dst,normalize='index')
transitions=pd.DataFrame(transition_rows);transitions.to_csv(OUTPUT_DIR/'view_transition_outcomes.csv',index=False)
display(transitions.sort_values('n',ascending=False).head(12))
if last_matrix is not None:
    fig,ax=plt.subplots(figsize=(9,7));im=ax.imshow(last_matrix,aspect='auto',vmin=0,vmax=1,cmap='Blues')
    ax.set_xticks(range(len(last_matrix.columns)),last_matrix.columns,rotation=90,fontsize=7)
    ax.set_yticks(range(len(last_matrix)),last_matrix.index,fontsize=7);fig.colorbar(im,ax=ax)
    ax.set(title=VIEW_METHOD+' — last transition',xlabel='To',ylabel='From');plt.tight_layout();plt.show()


## 5. Chốt shortlist và lưu

Bucket: sàng lọc theo tỷ trọng khách trong ô ít mẫu/event. KMeans: cùng reliability screen, không ép min_share của từng cluster. Merged groups: thêm min_share2% ở fit. Ngưỡng n100/event20/weak-share5%, rank0.85, AP-loss20%, Lift-loss5% là proposed screening để vận hành, không phải kiểm định thống kê.

Merged phải giữ tín hiệu so555 và so grid nguồn. Không tự chọn chỉ bằng silhouette hoặc stay rate. Nếu không có ứng viên đạt, không ép winner. Full ablation, history windows và lifecycle giữ riêng. Có thể đổi VIEW_METHOD thành KMeans_5/KMeans_6 để đọc profile và transition từ cùng code.


In [ ]:
MAX_AP_LOSS=.20;MAX_LIFT_LOSS=.05;MIN_RANK=.85
# Reliability across evaluation months, rather than requiring each bucket >=2%.
weak=outcomes.copy()
weak['weak_n']=np.where(weak.n.lt(MIN_GROUP_N)|weak.a.lt(MIN_GROUP_EVENTS),weak.n,0)
weak_month=weak.groupby(['method','month']).agg(weak_n=('weak_n','sum'),n=('n','sum'))
weak_month['weak_share']=weak_month.weak_n/weak_month.n
weak_max=weak_month.groupby('method').weak_share.max()
weak_month.to_csv(OUTPUT_DIR/'group_reliability_by_month.csv')
fit_profiles=profiles[profiles.month.eq(str(fit_month))]
min_share={name:g.n.min()/g.n.sum() for name,g in fit_profiles.groupby('method')}
selection=summary.loc[list(models)].copy();selection['min_fit_share']=pd.Series(min_share)
selection['AP_retained_source']=1.;selection['Lift_retained_source']=1.
for name in merge_maps:
    selection.loc[name,'AP_retained_source']=selection.loc[name,'AP']/summary.loc[MERGE_BASE,'AP']
    selection.loc[name,'Lift_retained_source']=selection.loc[name,'Lift10']/summary.loc[MERGE_BASE,'Lift10']
selection['weak_customer_share_max']=weak_max
selection['size_ok']=selection.weak_customer_share_max.le(MAX_WEAK_CUSTOMER_SHARE)
for name in merge_maps:
    selection.loc[name,'size_ok']=bool(selection.loc[name,'size_ok'] and selection.loc[name,'min_fit_share']>=MIN_MERGED_SHARE)
selection['eligible']=(selection.AP_retained_555.ge(1-MAX_AP_LOSS)&selection.Lift_retained_555.ge(1-MAX_LIFT_LOSS)&
    selection.AP_retained_source.ge(1-MAX_AP_LOSS)&selection.Lift_retained_source.ge(1-MAX_LIFT_LOSS)&
    selection.rank_stability_min.ge(MIN_RANK)&selection.size_ok)
selection.to_csv(OUTPUT_DIR/'selection_screen.csv')
accepted=selection[selection.eligible].sort_values(['groups','AP'],ascending=[True,False])
display(accepted.round(4))
if accepted.empty:print('No candidate passes the proposed screen; do not force a winner.')
config={'features':FEATURES,'fit_month':FIT_MONTH,'eval_months':EVAL_MONTHS,'preprocess':PREPROCESS,
    'seed':SEED,'k_options':K_OPTIONS,'silhouette_sample':SIL_SAMPLE,'history_start':str(fit_month),'fit_k_candidate':fit_k_candidate,'predictive_cohort':'HasEvent valid current+next products','geometry_cohort':'HasEvent valid current products','top10_ties':'expected fractional selection','rate_smoothing_pseudo_n':RATE_PRIOR_N,'merge_base':MERGE_BASE,'view_method':VIEW_METHOD,'boundary_sensitivity_methods':SENSITIVITY_METHODS,
    'source_manifest':str(candidates[0]),'thresholds':{'max_AP_loss':MAX_AP_LOSS,'max_Lift_loss':MAX_LIFT_LOSS,'min_rank':MIN_RANK,'min_merged_share':MIN_MERGED_SHARE,'min_group_n':MIN_GROUP_N,'min_group_events':MIN_GROUP_EVENTS,'max_weak_customer_share':MAX_WEAK_CUSTOMER_SHARE}}
(OUTPUT_DIR/'config.json').write_text(json.dumps(config,indent=2))
(OUTPUT_DIR/'bucket_specs.json').write_text(json.dumps({name:{f:e.tolist() for f,e in z.items()} for name,z in specs.items()},indent=2))
import joblib
joblib.dump({'scaler':scaler,'models':models,'specs':specs,'features':FEATURES,'preprocess':PREPROCESS},OUTPUT_DIR/'fitted_models.joblib')
manifest_out={'fit_month':FIT_MONTH,'eval_months':EVAL_MONTHS,'methods':list(models),'assignments':{name:[f'assignments_{name}_{m}.parquet' for m in [fit_month]+eval_months] for name in models} if SAVE_ASSIGNMENTS else {},'cohort':'HasEvent eligible only; join NoObservedEvent from upstream separately'}
(OUTPUT_DIR/'assignments_manifest.json').write_text(json.dumps(manifest_out,indent=2))
print('CSV / assignments / config saved:',OUTPUT_DIR)
